# FCN-Spielervergleich: Spiderplot-Tool

Mit diesem Tool können Spieler des FCN mit externen Spielern derselben Positionsgruppe verglichen werden.

## Bedienung

1. Referenzspieler auswählen.
2. Vergleichsspieler 1 auswählen.
3. Optional Vergleichsspieler 2 auswählen.
4. Auf „generieren“ klicken.
5. Optional den Plot als PNG/PDF speichern.

## Interpretation

Der FCN-Spieler ist immer auf 100 % normiert.  
Werte über 100 % bedeuten, dass der Vergleichsspieler in dieser Metrik über dem Referenzspieler liegt.  
Werte unter 100 % bedeuten, dass er darunter liegt.

Die kleinen Labels an der FCN-Linie zeigen die absoluten Referenzwerte. Sie beantworten also die Frage: „Wie viel ist 100 % in dieser Metrik?“

In [1]:
# =========================
# Imports
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import html
import textwrap
from pathlib import Path
from collections import defaultdict
from matplotlib import patches

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML as IPyHTML
except ImportError as exc:
    raise ImportError(
        "ipywidgets ist nicht installiert. Installiere es z. B. mit: pip install ipywidgets"
    ) from exc


In [12]:
# from google.colab import files
# import base64
# from pathlib import Path

# uploaded = files.upload()

# filename = next(iter(uploaded.keys()))
# excel_bytes = uploaded[filename]

# excel_b64 = base64.b64encode(excel_bytes).decode("ascii")

# # Zur Sicherheit nicht komplett printen, weil der String riesig ist.
# print("Datei:", filename)
# print("Base64-Länge:", len(excel_b64))
# print(excel_b64[:500])
# Path("embedded_excel_base64.txt").write_text(excel_b64)
# files.download("embedded_excel_base64.txt")

Saving fcn_spielerdiagramme_werte_mit_transfermarkt_groesse.xlsx to fcn_spielerdiagramme_werte_mit_transfermarkt_groesse.xlsx
Datei: fcn_spielerdiagramme_werte_mit_transfermarkt_groesse.xlsx
Base64-Länge: 134652
UEsDBBQABgAIAAAAIQDdImI6kAEAALAIAAATAAgCW0NvbnRlbnRfVHlwZXNdLnhtbCCiBAIooAACAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# =========================
# Datei laden
# =========================

SHEET_NAME = "Werte_lang"
TM_SHEET_NAME = "Transfermarkt"

import base64
from io import BytesIO

EMBEDDED_EXCEL_B64 = """
...
""".strip()

def get_embedded_excel_file():
    excel_bytes = base64.b64decode(EMBEDDED_EXCEL_B64)
    return BytesIO(excel_bytes)


In [3]:
# =========================
# Einstellungen
# =========================

# Wenn True, werden nicht explizit definierte Positionen zusätzlich einzeln als Gruppe angeboten.
INCLUDE_OTHER_POSITIONS = True

# Metriken, die nicht dargestellt werden sollen.
METRICS_TO_EXCLUDE = {"Fouls", "Fouls Drawn", "Cards"}

# Jugend-/Zweitteam-Spieler von Nürnberg ausschließen?
EXCLUDE_NUERNBERG_YOUTH = False

YOUTH_TEAM_PATTERNS = [
    r"\bii\b",         # Nürnberg II
    r"\bu[-\s]?17\b",  # Nürnberg U17, U-17, U 17
    r"\bu[-\s]?19\b",
    r"\bu[-\s]?21\b",
]

# Deckelung für die visuelle Darstellung.
CAP_PERCENT = 300
CAPPED_LABEL_BASE_OFFSET = 14
CAPPED_LABEL_LEVEL_GAP = 18

OUTPUT_DIR = Path("spider_plots")
OUTPUT_DIR.mkdir(exist_ok=True)

# Spaltennamen.
PLAYER_COL = "Spieler"
POSITION_COL = "Position"
METRIC_COL = "Metric"
VALUE_COL = "Wert"
LEAGUE_COL = "Liga"
TEAM_COL_CANDIDATES = ["Team", "Verein", "Club", "Mannschaft"]

# Positionsgruppen wie im ursprünglichen Notebook.
POSITION_GROUPS = {
    "LB": ["LB"],
    "RB": ["RB"],
    "LCB_RCB": ["LCB", "RCB", "CB"],
    "DMF_LCMF3_LDMF_RCMF3": ["DMF", "LCMF3", "LDMF", "RCMF3"],
    "AMF_LWF_RWF": ["AMF", "LWF", "RWF"],
    "LWF_RWF_CF": ["LWF", "RWF", "CF"],
}

# Logische Reihenfolge der Metriken.
METRIC_ORDER = [
    # Abschluss / Torgefahr
    "Shots",
    "Goals/Shot on Target %",
    "Non-Pen Goals",
    "npxG",
    "npxG per Shot",
    "Touches in Pen Box",

    # Kreativität / Chance Creation
    "Assists",
    "Second Assists",
    "Assists & 2nd/3rd Assists",
    "Shot Assists",
    "Expected Assists (xA)",
    "xA per Shot Assist",
    "Smart Passes",
    "Smart Pass %",
    "Crosses",
    "Cross Completion %",

    # Passspiel / Ballzirkulation / Progression
    "Received Passes",
    "Passes",
    "Short & Med Pass %",
    "% of Passes Being Short",
    "% of Passes Being Lateral",
    "Long Pass %",
    "Long Pass Cmp %",
    "Prog. Passes",
    "Prog. Carries",

    # Dribbling / Balltransport
    "Acceleration with Ball",
    "Dribble Success %",

    # Defensivarbeit
    "Defensive Actions",
    "Defensive Duels Won %",
    "Tackles (pAdj)",
    "Interceptions (pAdj)",
    "Tackles & Int (pAdj)",
    "Shot Blocks",
    "Aerial Duels Won",
    "Aerial Win %",

    # Torwart-spezifisch
    "Save %",
    "Shots Against",
    "Goals Conceded",
    "Prevented Goals",
    "Goals Prevented %",
    "Coming Off Line",
]

METRIC_ORDER_MAP = {metric: i for i, metric in enumerate(METRIC_ORDER)}


In [8]:
# =========================
# Hilfsfunktionen: Daten, Gruppen, relative Werte, Transfermarkt, Layout
# =========================

FCN_RED = "#8B0000"
FCN_RED_LIGHT = "#C62828"
FCN_BLACK = "#1F1F1F"
FCN_GREY = "#5F6368"
FCN_BG = "#FAF7F7"
PANEL_BG = "#FBFBFC"
PANEL_BORDER = "#D9D9DE"
NON_FCN_COLORS = ["#F39C12", "#1B9E77", "#4C78A8", "#7F7F7F"]
FCN_PLAYER_COLORS = [FCN_RED, FCN_BLACK, FCN_RED_LIGHT]

def wrap_text(text, width=34, break_long_words=False):
    return "\n".join(
        textwrap.wrap(
            str(text),
            width=width,
            break_long_words=break_long_words,
            break_on_hyphens=False,
        )
    )

def sanitize_filename(text):
    text = re.sub(r"[^\w\s-]", "", str(text), flags=re.UNICODE)
    text = re.sub(r"[-\s]+", "_", text)
    return text.strip("_")[:120]


def is_nuernberg_text(text):
    text = str(text).lower()
    patterns = [
        "nürnberg",
        "nuernberg",
        "nurnberg",
        "1. fc nürnberg",
        "1. fc nuernberg",
        "1. fc nurnberg",
        "fcn",
        "1. fcn",
    ]
    return any(p in text for p in patterns)


def is_nuernberg_youth_team(text):
    text = str(text).lower()
    if not is_nuernberg_text(text):
        return False
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in YOUTH_TEAM_PATTERNS)


def sort_metrics_logically(metrics):
    return sorted(metrics, key=lambda m: (METRIC_ORDER_MAP.get(m, 10_000), m))


def first_non_empty(values):
    for value in values:
        if pd.isna(value):
            continue
        if isinstance(value, str) and value.strip() == "":
            continue
        return value
    return np.nan


def format_display_value(value):
    if pd.isna(value):
        return "k. A."
    if isinstance(value, pd.Timestamp):
        return value.strftime("%d.%m.%Y")
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return "k. A."
    return text


def compact_url(url):
    url = format_display_value(url)
    if url == "k. A.":
        return url
    url = re.sub(r"^https?://", "", url)
    return url.replace("www.", "")


def detect_first_existing_column(df_like, candidates):
    return next((c for c in candidates if c in df_like.columns), None)


def wrap_metric_label(label, width=16):
    label = str(label)
    if len(label) <= width:
        return label

    words = label.split()
    if len(words) == 1:
        return textwrap.fill(label, width=width)

    wrapped = textwrap.fill(label, width=width, break_long_words=False, break_on_hyphens=False)
    return wrapped


def wrap_card_line(text, width=34):
    text = str(text)
    return textwrap.fill(text, width=width, break_long_words=False, break_on_hyphens=False)


def load_and_prepare_data(file_path, sheet_name):
    raw_df = pd.read_excel(file_path, sheet_name=sheet_name)

    required_cols = [PLAYER_COL, POSITION_COL, METRIC_COL, VALUE_COL, LEAGUE_COL]
    missing_cols = [c for c in required_cols if c not in raw_df.columns]
    if missing_cols:
        raise ValueError(f"Diese Spalten fehlen im Sheet: {missing_cols}")

    detected_team_col = next((c for c in TEAM_COL_CANDIDATES if c in raw_df.columns), None)

    keep_cols = required_cols.copy()
    if detected_team_col is not None:
        keep_cols.append(detected_team_col)

    clean_df = raw_df[keep_cols].copy()
    clean_df = clean_df.dropna(subset=[PLAYER_COL, POSITION_COL, METRIC_COL, VALUE_COL])
    clean_df[PLAYER_COL] = clean_df[PLAYER_COL].astype(str).str.strip()
    clean_df[POSITION_COL] = clean_df[POSITION_COL].astype(str).str.strip()
    clean_df[METRIC_COL] = clean_df[METRIC_COL].astype(str).str.strip()
    clean_df[VALUE_COL] = pd.to_numeric(clean_df[VALUE_COL], errors="coerce")
    clean_df = clean_df.dropna(subset=[VALUE_COL])

    clean_df = clean_df[~clean_df[METRIC_COL].isin(METRICS_TO_EXCLUDE)].copy()

    if EXCLUDE_NUERNBERG_YOUTH:
        if detected_team_col is None:
            print("Warnung: Kein Team-Feld gefunden, Jugend-/Zweitteam-Filter kann nicht angewendet werden.")
        else:
            before_players = clean_df[PLAYER_COL].nunique()
            clean_df = clean_df[~clean_df[detected_team_col].apply(is_nuernberg_youth_team)].copy()
            after_players = clean_df[PLAYER_COL].nunique()
            print(f"Jugend-/Zweitteam-Filter aktiv: {before_players - after_players} Spieler entfernt.")

    group_cols = [PLAYER_COL, POSITION_COL, METRIC_COL, LEAGUE_COL]
    if detected_team_col is not None:
        group_cols.append(detected_team_col)

    clean_df = clean_df.groupby(group_cols, as_index=False)[VALUE_COL].mean()

    return clean_df, detected_team_col


def load_transfermarkt_data(file_path, sheet_name):
    xls = pd.ExcelFile(file_path)
    if sheet_name not in xls.sheet_names:
        print(f"Hinweis: Transfermarkt-Sheet '{sheet_name}' wurde nicht gefunden.")
        empty = pd.DataFrame(
            columns=[PLAYER_COL, "tm_team", "tm_market_value", "tm_contract_until", "tm_height", "tm_profile_url"]
        ).set_index(PLAYER_COL)
        return empty, {}

    raw_tm_df = pd.read_excel(file_path, sheet_name=sheet_name)
    if PLAYER_COL not in raw_tm_df.columns:
        print(f"Hinweis: Im Transfermarkt-Sheet fehlt die Spalte '{PLAYER_COL}'.")
        empty = pd.DataFrame(
            columns=[PLAYER_COL, "tm_team", "tm_market_value", "tm_contract_until", "tm_height", "tm_profile_url"]
        ).set_index(PLAYER_COL)
        return empty, {}

    raw_tm_df = raw_tm_df.copy()
    raw_tm_df[PLAYER_COL] = raw_tm_df[PLAYER_COL].astype(str).str.strip()
    raw_tm_df = raw_tm_df[raw_tm_df[PLAYER_COL] != ""]

    column_candidates = {
        "tm_team": ["TM aktueller Verein", "Team", "Team in Ausgangstabelle"],
        "tm_market_value": ["TM Marktwert"],
        "tm_contract_until": ["TM Vertrag bis"],
        "tm_height": ["Größe", "Groesse", "TM Größe", "TM Groesse"],
        "tm_profile_url": ["TM Profil-URL", "Profil-URL", "Transfermarkt-Profil-URL"],
    }

    detected_columns = {
        key: detect_first_existing_column(raw_tm_df, candidates)
        for key, candidates in column_candidates.items()
    }

    rows = []
    for player_name, player_rows in raw_tm_df.groupby(PLAYER_COL, sort=True):
        row = {PLAYER_COL: player_name}
        for target_col, source_col in detected_columns.items():
            row[target_col] = first_non_empty(player_rows[source_col]) if source_col else np.nan
        rows.append(row)

    if not rows:
        empty = pd.DataFrame(
            columns=[PLAYER_COL, "tm_team", "tm_market_value", "tm_contract_until", "tm_height", "tm_profile_url"]
        ).set_index(PLAYER_COL)
        return empty, detected_columns

    tm_df_clean = pd.DataFrame(rows).set_index(PLAYER_COL)
    return tm_df_clean, detected_columns


def build_plot_groups():
    groups = {}
    used_positions = set()

    for group_name, positions in POSITION_GROUPS.items():
        group_df = df[df[POSITION_COL].isin(positions)].copy()
        if not group_df.empty:
            groups[group_name] = group_df
            used_positions.update(positions)

    if INCLUDE_OTHER_POSITIONS:
        remaining_positions = sorted(
            p for p in df[POSITION_COL].dropna().unique()
            if p not in used_positions
        )
        for pos in remaining_positions:
            group_df = df[df[POSITION_COL] == pos].copy()
            if not group_df.empty:
                groups[pos] = group_df

    return groups


def get_group_df_by_name(group_name):
    if group_name in POSITION_GROUPS:
        positions = POSITION_GROUPS[group_name]
        return df[df[POSITION_COL].isin(positions)].copy()
    return df[df[POSITION_COL] == group_name].copy()


def get_all_available_group_names():
    return list(plot_groups.keys())


def get_candidate_groups_for_player(player_name):
    player_df = df[df[PLAYER_COL] == player_name].copy()
    if player_df.empty:
        return []

    candidate_groups = []
    for group_name in get_all_available_group_names():
        group_df = get_group_df_by_name(group_name)
        if player_name in set(group_df[PLAYER_COL].unique()):
            candidate_groups.append(group_name)

    return candidate_groups


def prepare_relative_values(group_df, reference_player):
    values = group_df.pivot_table(
        index=PLAYER_COL,
        columns=METRIC_COL,
        values=VALUE_COL,
        aggfunc="mean",
    )
    values = values.dropna(axis=1, how="all")

    if reference_player not in values.index:
        raise ValueError(f"Referenzspieler '{reference_player}' ist nicht in dieser Gruppe enthalten.")

    ref_values = values.loc[reference_player]
    usable_metrics = ref_values[(ref_values.notna()) & (ref_values != 0)].index.tolist()
    usable_metrics = sort_metrics_logically(usable_metrics)

    values = values[usable_metrics]
    ref_values = ref_values[usable_metrics]
    relative_values = values.divide(ref_values, axis=1) * 100

    return relative_values, values, ref_values


def get_nuernberg_players():
    if team_col is None:
        print("Warnung: Kein Team-Feld gefunden. Referenzliste fällt auf alle Spieler zurück.")
        candidate_players = sorted(df[PLAYER_COL].dropna().unique())
    else:
        candidate_players = []
        for player, player_df in df.groupby(PLAYER_COL):
            teams = player_df[team_col].dropna().astype(str).unique().tolist()
            if any(is_nuernberg_text(team) for team in teams):
                candidate_players.append(player)
        candidate_players = sorted(candidate_players)

    return [p for p in candidate_players if get_candidate_groups_for_player(p)]


def get_player_team_from_main_data(player_name):
    if team_col is None:
        return np.nan
    player_rows = df[df[PLAYER_COL] == player_name]
    if player_rows.empty:
        return np.nan
    team_values = player_rows[team_col].dropna().astype(str).unique().tolist()
    return team_values[0] if team_values else np.nan


def get_player_league_from_main_data(player_name):
    player_rows = df[df[PLAYER_COL] == player_name]
    if player_rows.empty:
        return np.nan
    league_values = player_rows[LEAGUE_COL].dropna().astype(str).unique().tolist()
    return league_values[0] if league_values else np.nan


def build_legend_label(player_name):
    league = format_display_value(get_player_league_from_main_data(player_name))
    if league == "k. A.":
        return player_name
    return f"{player_name} | {league}"


def is_fcn_player(player_name):
    candidate_texts = []

    team_from_main = get_player_team_from_main_data(player_name)
    if pd.notna(team_from_main):
        candidate_texts.append(team_from_main)

    if 'tm_info_df' in globals() and not tm_info_df.empty and player_name in tm_info_df.index:
        tm_team = tm_info_df.loc[player_name, 'tm_team']
        if pd.notna(tm_team):
            candidate_texts.append(tm_team)

    return any(is_nuernberg_text(text) for text in candidate_texts)


def get_transfermarkt_profile(player_name):
    fallback_team = get_player_team_from_main_data(player_name)

    profile = {
        'Team': format_display_value(fallback_team),
        'TM Marktwert': 'k. A.',
        'TM Vertrag bis': 'k. A.',
        'Größe': 'k. A.',
        'Profil-URL': 'k. A.',
    }

    if 'tm_info_df' not in globals() or tm_info_df.empty:
        return profile

    if player_name not in tm_info_df.index:
        return profile

    player_row = tm_info_df.loc[player_name]
    if isinstance(player_row, pd.DataFrame):
        player_row = player_row.iloc[0]

    team_value = player_row.get('tm_team', np.nan)
    if pd.notna(team_value):
        profile['Team'] = format_display_value(team_value)

    profile['TM Marktwert'] = format_display_value(player_row.get('tm_market_value', np.nan))
    profile['TM Vertrag bis'] = format_display_value(player_row.get('tm_contract_until', np.nan))
    profile['Größe'] = format_display_value(player_row.get('tm_height', np.nan))
    profile['Profil-URL'] = format_display_value(player_row.get('tm_profile_url', np.nan))

    return profile


def build_clickable_links_html(non_fcn_players):
    if not non_fcn_players:
        return ""

    blocks = []
    for player in non_fcn_players:
        profile = get_transfermarkt_profile(player)
        url = profile.get('Profil-URL', 'k. A.')
        if url == 'k. A.':
            blocks.append(
                f"<li><strong>{html.escape(player)}</strong>: kein Transfermarkt-Link verfügbar</li>"
            )
        else:
            safe_url = html.escape(url, quote=True)
            safe_name = html.escape(player)
            blocks.append(
                f'<li><strong>{safe_name}</strong>: <a href="{safe_url}" target="_blank" rel="noopener noreferrer">Transfermarkt-Profil öffnen ↗</a></li>'
            )

    return f'''
    <div style="
        margin-top:10px;
        background:{FCN_BG};
        border:1px solid {PANEL_BORDER};
        border-left:6px solid {FCN_RED};
        border-radius:12px;
        padding:12px 16px;
        font-family:Arial, Helvetica, sans-serif;
        width:1180px;
    ">
        <div style="font-size:16px;font-weight:700;color:{FCN_RED};margin-bottom:8px;">Klickbare Transfermarkt-Links</div>
        <ul style="margin:0;padding-left:18px;line-height:1.7;">
            {''.join(blocks)}
        </ul>
    </div>
    '''


def style_for_player(player_name, fcn_counter, non_fcn_counter):
    if is_fcn_player(player_name):
        color = FCN_PLAYER_COLORS[min(fcn_counter, len(FCN_PLAYER_COLORS) - 1)]
        fcn_counter += 1
    else:
        color = NON_FCN_COLORS[non_fcn_counter % len(NON_FCN_COLORS)]
        non_fcn_counter += 1
    return color, fcn_counter, non_fcn_counter


def clip_for_plot(series, cap=CAP_PERCENT):
    arr = series.to_numpy(dtype=float)
    return np.where(np.isnan(arr), np.nan, np.minimum(arr, cap))


def round_up_to_step(x, step=25):
    return int(np.ceil(x / step) * step)


def determine_dynamic_radial_limit(relative_values, cap=CAP_PERCENT, step=25):
    arr = relative_values.to_numpy(dtype=float)
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        return 100

    displayed_max = np.min([np.nanmax(arr), cap])
    if displayed_max <= 0:
        return 100

    if displayed_max % step == 0:
        axis_limit = displayed_max + step
    else:
        axis_limit = round_up_to_step(displayed_max, step)

    axis_limit = min(axis_limit, cap)
    axis_limit = max(axis_limit, 100)

    return int(axis_limit)


def build_radial_ticks(axis_limit, step=25):
    return list(range(step, int(axis_limit) + 1, step))


def format_reference_raw_value(metric, value):
    """Formatiert den absoluten Rohwert des Referenzspielers für kleine Labels am 100%-Ring."""
    if pd.isna(value):
        return ""

    value = float(value)
    suffix = "%" if "%" in str(metric) else ""

    if suffix:
        if abs(value) >= 10:
            text = f"{value:.0f}" if abs(value - round(value)) < 0.05 else f"{value:.1f}"
        else:
            text = f"{value:.1f}"
    else:
        if abs(value) >= 100:
            text = f"{value:.0f}"
        elif abs(value) >= 10:
            text = f"{value:.1f}"
        elif abs(value) >= 1:
            text = f"{value:.2f}".rstrip("0").rstrip(".")
        else:
            text = f"{value:.2f}" if abs(value) >= 0.1 else f"{value:.3f}"
            text = text.rstrip("0").rstrip(".")

    return f"{text}{suffix}"


def place_reference_value_annotations(ax, angles, metrics, ref_values, base_radius=100, axis_limit=100):
    """Beschriftet die Datenpunkte des Referenzspielers mit dessen absoluten Rohwerten."""
    if ref_values is None or len(metrics) == 0:
        return

    max_label_radius = max(axis_limit, base_radius) + 20

    for idx, (angle, metric) in enumerate(zip(angles[:-1], metrics)):
        if metric not in ref_values.index:
            continue

        label = format_reference_raw_value(metric, ref_values.loc[metric])
        if not label:
            continue

        # Kleine Radial-Staffelung verhindert, dass benachbarte Labels direkt aufeinander liegen.
        radial_offset = 8 if idx % 2 == 0 else -8
        label_radius = base_radius + radial_offset
        label_radius = min(max(label_radius, 18), max_label_radius)

        cos_a = np.cos(angle)
        if cos_a > 0.35:
            ha = "left"
        elif cos_a < -0.35:
            ha = "right"
        else:
            ha = "center"

        ax.annotate(
            label,
            xy=(angle, base_radius),
            xytext=(angle, label_radius),
            textcoords="data",
            ha=ha,
            va="center",
            fontsize=7.4,
            fontweight="bold",
            color=FCN_RED,
            clip_on=False,
            bbox=dict(
                boxstyle="round,pad=0.22",
                facecolor="white",
                edgecolor=FCN_RED,
                linewidth=0.65,
                alpha=0.88,
            ),
            zorder=25,
        )


def place_capped_annotations(ax, capped_annotations, cap=CAP_PERCENT, axis_limit=None):
    if not capped_annotations:
        return

    if axis_limit is None:
        axis_limit = cap

    grouped = defaultdict(list)
    for item in capped_annotations:
        grouped[item["metric_idx"]].append(item)

    angle_jitter = np.deg2rad(2.0)

    for metric_idx, items in grouped.items():
        items = sorted(items, key=lambda x: x["true_value"])

        for level, item in enumerate(items):
            base_angle = item["angle"]
            true_value = item["true_value"]
            color = item["color"]

            if level == 0:
                jitter_factor = 0
            elif level % 2 == 1:
                jitter_factor = (level + 1) // 2
            else:
                jitter_factor = -(level // 2)

            label_angle = base_angle + jitter_factor * angle_jitter
            label_radius = max(axis_limit, cap) + CAPPED_LABEL_BASE_OFFSET + level * CAPPED_LABEL_LEVEL_GAP

            cos_a = np.cos(label_angle)
            if cos_a > 0.25:
                ha = "left"
            elif cos_a < -0.25:
                ha = "right"
            else:
                ha = "center"

            ax.annotate(
                f"{true_value:.0f}%",
                xy=(base_angle, cap),
                xytext=(label_angle, label_radius),
                textcoords="data",
                ha=ha,
                va="center",
                fontsize=8,
                fontweight="bold",
                color=color,
                clip_on=False,
                arrowprops=dict(
                    arrowstyle="-",
                    color=color,
                    lw=0.8,
                    alpha=0.75,
                    shrinkA=0,
                    shrinkB=0,
                ),
                zorder=20,
            )


def draw_player_card(ax, x, y_top, width, height, player_name):
    profile = get_transfermarkt_profile(player_name)

    card = patches.FancyBboxPatch(
        (x, y_top - height),
        width,
        height,
        boxstyle="round,pad=0.012,rounding_size=0.02",
        linewidth=1.0,
        edgecolor=PANEL_BORDER,
        facecolor="white",
        transform=ax.transAxes,
    )
    ax.add_patch(card)

    accent = patches.FancyBboxPatch(
        (x, y_top - 0.035),
        width,
        0.02,
        boxstyle="round,pad=0,rounding_size=0.02",
        linewidth=0,
        facecolor=FCN_RED,
        transform=ax.transAxes,
    )
    ax.add_patch(accent)

    ax.text(
        x + 0.03,
        y_top - 0.06,
        player_name,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=11.5,
        fontweight="bold",
        color=FCN_BLACK,
    )

    lines = [
        f"Team: {profile['Team']}",
        f"TM Marktwert: {profile['TM Marktwert']}",
        f"TM Vertrag bis: {profile['TM Vertrag bis']}",
        f"Größe: {profile['Größe']}",
        # f"TM Profil: {compact_url(profile['Profil-URL'])}",
    ]
    wrapped_lines = []
    for line in lines:
        wrapped_lines.extend(wrap_card_line(line, width=30).split("\n"))

    # url_line = f"TM Profil: {compact_url(profile['Profil-URL'])}"
    # wrapped_url = wrap_text(url_line, width=32, break_long_words=True)

    # wrapped_lines.extend(wrap_text(
    #     f"TM Profil: {compact_url(profile['Profil-URL'])}",
    #     width=32,
    #     break_long_words=True,
    # ).split("\n"))

    ax.text(
        x + 0.03,
        y_top - 0.12,
        "\n".join(wrapped_lines),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9.4,
        color=FCN_BLACK,
        linespacing=1.45,
    )


def draw_info_panel(info_ax, legend_items, non_fcn_players):
    info_ax.axis("off")
    info_ax.set_xlim(0, 1)
    info_ax.set_ylim(0, 1)
    info_ax.set_facecolor(PANEL_BG)

    outer = patches.FancyBboxPatch(
        (0.02, 0.02),
        0.96,
        0.96,
        boxstyle="round,pad=0.012,rounding_size=0.02",
        linewidth=1.1,
        edgecolor=PANEL_BORDER,
        facecolor=PANEL_BG,
        transform=info_ax.transAxes,
    )
    info_ax.add_patch(outer)

    info_ax.text(0.07, 0.95, "Infobereich", transform=info_ax.transAxes,
                 ha="left", va="top", fontsize=16, fontweight="bold", color=FCN_RED)

    info_ax.text(0.07, 0.90, "Legende", transform=info_ax.transAxes,
                 ha="left", va="top", fontsize=12.5, fontweight="bold", color=FCN_BLACK)

    y = 0.855
    for item in legend_items:
        info_ax.plot([0.08, 0.18], [y, y], transform=info_ax.transAxes,
                     color=item['color'], linewidth=2.6, linestyle=item['linestyle'], solid_capstyle='round')

        legend_label = wrap_text(item["label"], width=36)
        line_count = legend_label.count("\n") + 1
        info_ax.text(0.21, y, legend_label, transform=info_ax.transAxes,
                     ha="left", va="center", fontsize=9.6, color=FCN_BLACK, linespacing=1.25)
        y -= 0.04

    info_ax.text(
        0.07,
        y - 0.005,
        "Labels an der FCN-Linie = absolute Werte",
        transform=info_ax.transAxes,
        ha="left",
        va="top",
        fontsize=8.8,
        color=FCN_GREY,
        wrap=True,
    )

    steckbrief_heading_y = y - 0.045

    info_ax.text(
        0.07,
        steckbrief_heading_y,
        "Steckbrief(e)",
        transform=info_ax.transAxes,
        ha="left",
        va="top",
        fontsize=12.5,
        fontweight="bold",
        color=FCN_BLACK,
    )

    card_start_y = steckbrief_heading_y - 0.06

    if non_fcn_players:
        players_to_show = non_fcn_players[:3]
        n_cards = len(players_to_show)

        gap = 0.025
        bottom_padding = 0.045

        available_height = card_start_y - bottom_padding - (n_cards - 1) * gap

        # Kein harter 0.22-Deckel mehr.
        # Die Karten werden automatisch so hoch wie möglich, ohne sich zu überlappen.
        card_height = available_height / n_cards

        # Sicherheitsdeckel: bei nur einem Steckbrief nicht unnötig riesig.
        card_height = min(card_height, 0.30)

        current_y = card_start_y

        for player in players_to_show:
            draw_player_card(
                info_ax,
                x=0.06,
                y_top=current_y,
                width=0.88,
                height=card_height,
                player_name=player,
            )
            current_y -= card_height + gap
    else:
        note = patches.FancyBboxPatch(
            (0.06, card_start_y - 0.16), 0.88, 0.12,
            boxstyle="round,pad=0.012,rounding_size=0.02",
            linewidth=1.0, edgecolor=PANEL_BORDER, facecolor="white", transform=info_ax.transAxes
        )
        info_ax.add_patch(note)
        info_ax.text(
            0.09, card_start_y - 0.06,
            "Alle dargestellten Spieler spielen beim FCN – daher sind keine externen Steckbriefe nötig.",
            transform=info_ax.transAxes, ha="left", va="top", fontsize=9.6, color=FCN_BLACK,
            wrap=True,
        )


In [9]:
# Daten einlesen und Gruppen vorbereiten
# =========================

df, team_col = load_and_prepare_data(
    get_embedded_excel_file(),
    SHEET_NAME,
)

tm_info_df, tm_detected_cols = load_transfermarkt_data(
    get_embedded_excel_file(),
    TM_SHEET_NAME,
)
plot_groups = build_plot_groups()
nuernberg_players = get_nuernberg_players()

print(f"Daten geladen: {df[PLAYER_COL].nunique()} Spieler, {df[METRIC_COL].nunique()} Metriken")
print(f"Team-Spalte: {team_col if team_col is not None else 'nicht gefunden'}")
print(f"Verfügbare Gruppen: {len(plot_groups)}")
print(f"Nürnberg-Referenzspieler in der GUI: {len(nuernberg_players)}")

if tm_info_df.empty:
    print("Transfermarkt-Daten: kein nutzbares Transfermarkt-Sheet gefunden oder Sheet ist leer.")
else:
    available_tm_fields = [
        name for name, source_col in tm_detected_cols.items()
        if source_col is not None
    ]
    print(f"Transfermarkt-Daten geladen für: {len(tm_info_df)} Spieler")
    print(f"Verfügbare TM-Felder: {', '.join(available_tm_fields) if available_tm_fields else 'keine'}")

if not nuernberg_players:
    raise ValueError(
        "Es wurden keine Nürnberg-Spieler gefunden. Prüfe die Team-Spalte oder die Nürnberg-Schreibweise."
    )


Daten geladen: 37 Spieler, 41 Metriken
Team-Spalte: Team
Verfügbare Gruppen: 9
Nürnberg-Referenzspieler in der GUI: 19
Transfermarkt-Daten geladen für: 18 Spieler
Verfügbare TM-Felder: tm_team, tm_market_value, tm_contract_until, tm_height, tm_profile_url


In [10]:
# =========================
# Spiderplot-Funktion für GUI: Referenz + 1 oder 2 Vergleichsspieler
# =========================

def make_spider_plot_comparison(reference_player, comparison_players, explicit_group, save_plot=False):
    comparison_players = [p for p in comparison_players if p is not None and p != ""]

    if not reference_player:
        raise ValueError("Bitte einen Referenzspieler auswählen.")
    if len(comparison_players) < 1:
        raise ValueError("Bitte mindestens einen Vergleichsspieler auswählen.")
    if reference_player in comparison_players:
        raise ValueError("Referenzspieler und Vergleichsspieler müssen unterschiedlich sein.")
    if len(set(comparison_players)) != len(comparison_players):
        raise ValueError("Vergleichsspieler dürfen nicht doppelt ausgewählt werden.")
    if explicit_group is None:
        raise ValueError("Keine Gruppe ausgewählt. Bitte Vergleichsspieler 1 wählen.")

    group_df_full = get_group_df_by_name(explicit_group).copy()
    group_players = set(group_df_full[PLAYER_COL].unique())

    missing = [p for p in [reference_player] + comparison_players if p not in group_players]
    if missing:
        raise ValueError(f"Diese Spieler sind nicht in der Gruppe '{explicit_group}': {missing}")

    player_order = [reference_player] + comparison_players
    group_df = group_df_full[group_df_full[PLAYER_COL].isin(player_order)].copy()

    relative_values, raw_values, ref_values = prepare_relative_values(
        group_df=group_df,
        reference_player=reference_player,
    )

    relative_values = relative_values.reindex(player_order)
    metrics = relative_values.columns.tolist()

    if len(metrics) < 3:
        raise ValueError(
            f"Für den Plot gibt es weniger als 3 nutzbare Metriken in der Gruppe '{explicit_group}'."
        )

    non_fcn_players = [player for player in player_order if not is_fcn_player(player)]
    legend_items = []
    capped_annotations = []

    fig = plt.figure(figsize=(12.5, 8.8), constrained_layout=True)
    gs = fig.add_gridspec(1, 2, width_ratios=[3.45, 1.45])
    ax = fig.add_subplot(gs[0, 0], polar=True)
    info_ax = fig.add_subplot(gs[0, 1])

    fig.patch.set_facecolor("white")
    ax.set_facecolor(FCN_BG)

    n_metrics = len(metrics)
    angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
    angles += angles[:1]

    fcn_counter = 0
    non_fcn_counter = 0

    for idx, player in enumerate(player_order):
        true_vals = relative_values.loc[player]
        clipped_vals = clip_for_plot(true_vals, cap=CAP_PERCENT)
        vals_closed = clipped_vals.tolist() + [clipped_vals[0]]

        if idx == 0:
            linewidth = 2.8
            linestyle = "-"
            alpha_fill = 0.08
        elif idx == 1:
            linewidth = 2.3
            linestyle = "--"
            alpha_fill = 0.05
        else:
            linewidth = 2.2
            linestyle = ":"
            alpha_fill = 0.04

        color, fcn_counter, non_fcn_counter = style_for_player(player, fcn_counter, non_fcn_counter)

        line, = ax.plot(
            angles,
            vals_closed,
            linewidth=linewidth,
            linestyle=linestyle,
            color=color,
        )

        legend_items.append({
            "player": player,
            "label": build_legend_label(player),
            "color": color,
            "linestyle": linestyle,
        })

        if not np.isnan(clipped_vals).any():
            ax.fill(angles, vals_closed, alpha=alpha_fill, color=color)

        for metric_idx, angle in enumerate(angles[:-1]):
            true_value = true_vals.iloc[metric_idx]
            if pd.notna(true_value) and true_value > CAP_PERCENT:
                capped_annotations.append({
                    "metric_idx": metric_idx,
                    "metric": metrics[metric_idx],
                    "angle": angle,
                    "true_value": true_value,
                    "player": player,
                    "color": color,
                })

    axis_limit = determine_dynamic_radial_limit(relative_values=relative_values, cap=CAP_PERCENT, step=25)

    if capped_annotations:
        counts_by_metric = defaultdict(int)
        for item in capped_annotations:
            counts_by_metric[item["metric_idx"]] += 1
        max_stack = max(counts_by_metric.values())
        ylim_top = max(axis_limit, CAP_PERCENT) + CAPPED_LABEL_BASE_OFFSET + (max_stack - 1) * CAPPED_LABEL_LEVEL_GAP + 35
    else:
        ylim_top = axis_limit

    ax.set_ylim(0, ylim_top)
    ax.grid(alpha=0.45)

    yticks = build_radial_ticks(axis_limit, step=25)
    ax.set_yticks(yticks)
    ax.set_yticklabels([f"{y}%" for y in yticks], fontsize=9, color=FCN_BLACK)
    ax.set_rlabel_position(142)

    place_reference_value_annotations(
        ax=ax,
        angles=angles,
        metrics=metrics,
        ref_values=ref_values,
        base_radius=100,
        axis_limit=axis_limit,
    )

    place_capped_annotations(
        ax=ax,
        capped_annotations=capped_annotations,
        cap=CAP_PERCENT,
        axis_limit=axis_limit,
    )

    wrapped_metrics = [wrap_metric_label(metric, width=16) for metric in metrics]
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(wrapped_metrics, fontsize=9.2, color=FCN_BLACK)
    ax.tick_params(axis="x", pad=10)

    positions = sorted(group_df_full[POSITION_COL].dropna().unique())
    comparison_text = " vs. ".join(player_order)

    title = (
        f"{comparison_text}\n"
        f"Gruppe: {explicit_group} | Positionen: {', '.join(positions)}\n"
        "Werte jeweils pro90, relativ bezogen auf Referenzspieler (= 100%)"
    )
    ax.set_title(title, fontsize=16.5, pad=34, color=FCN_BLACK)

    draw_info_panel(info_ax, legend_items=legend_items, non_fcn_players=non_fcn_players)

    if save_plot:
        filename = sanitize_filename(f"gui_tm_fcn_{explicit_group}_{'_vs_'.join(player_order)}")
        png_path = OUTPUT_DIR / f"{filename}.png"
        pdf_path = OUTPUT_DIR / f"{filename}.pdf"
        fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
        fig.savefig(pdf_path, bbox_inches="tight", facecolor=fig.get_facecolor())
        print(f"Gespeichert: {png_path}")
        print(f"Gespeichert: {pdf_path}")

    return {
        "group": explicit_group,
        "figure": fig,
        "non_fcn_players": non_fcn_players,
    }


In [11]:
# =========================
# GUI
# =========================

SEPARATOR = "|||"


def encode_selection(group_name, player_name):
    return f"{group_name}{SEPARATOR}{player_name}"


def decode_selection(value):
    if value in (None, ""):
        return None, None
    group_name, player_name = value.split(SEPARATOR, 1)
    return group_name, player_name


def comparison_options_for_reference(reference_player):
    options = [("— bitte wählen —", None)]
    seen = set()

    for group_name in get_candidate_groups_for_player(reference_player):
        group_df = get_group_df_by_name(group_name)
        players = sorted(p for p in group_df[PLAYER_COL].dropna().unique() if p != reference_player)

        for player in players:
            value = encode_selection(group_name, player)
            if value in seen:
                continue
            seen.add(value)
            label = f"{player} [{group_name}]"
            options.append((label, value))

    return options


def comparison2_options_for_group(reference_player, comparison1_player, group_name):
    options = [("— kein zweiter Vergleichsspieler —", None)]

    if group_name is None:
        return options

    group_df = get_group_df_by_name(group_name)
    players = sorted(
        p for p in group_df[PLAYER_COL].dropna().unique()
        if p not in {reference_player, comparison1_player}
    )

    for player in players:
        options.append((player, encode_selection(group_name, player)))

    return options


reference_dropdown = widgets.Dropdown(
    options=[(p, p) for p in nuernberg_players],
    description="Referenz",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "90px"},
)

comparison1_dropdown = widgets.Dropdown(
    options=[],
    description="Vergleich 1",
    layout=widgets.Layout(width="360px"),
    style={"description_width": "90px"},
)

comparison2_dropdown = widgets.Dropdown(
    options=[("— kein zweiter Vergleichsspieler —", None)],
    description="Vergleich 2",
    layout=widgets.Layout(width="360px"),
    style={"description_width": "90px"},
    disabled=True,
)

generate_button = widgets.Button(
    description="generieren",
    button_style="success",
    icon="line-chart",
    layout=widgets.Layout(width="160px"),
    disabled=True,
)

save_checkbox = widgets.Checkbox(
    value=False,
    description="Plot zusätzlich als PNG/PDF speichern",
    indent=False,
    layout=widgets.Layout(width="280px"),
)

status_output = widgets.Output()
plot_output = widgets.Output()
links_output = widgets.Output()


def update_generate_button_state():
    generate_button.disabled = not (reference_dropdown.value and comparison1_dropdown.value)


def clear_outputs_after_selection_change():
    with plot_output:
        clear_output(wait=True)
    with links_output:
        clear_output(wait=True)


def refresh_comparison1_options(*args):
    reference_player = reference_dropdown.value
    comparison1_dropdown.options = comparison_options_for_reference(reference_player)
    comparison1_dropdown.value = None
    comparison2_dropdown.options = [("— kein zweiter Vergleichsspieler —", None)]
    comparison2_dropdown.value = None
    comparison2_dropdown.disabled = True
    clear_outputs_after_selection_change()
    update_generate_button_state()

    with status_output:
        clear_output(wait=True)
        groups = get_candidate_groups_for_player(reference_player)
        print(f"Referenzspieler: {reference_player}")
        print(f"Verfügbare Gruppe(n): {', '.join(groups)}")
        print("Wähle Vergleich 1; dadurch wird die Gruppe für den Plot festgelegt.")


def refresh_comparison2_options(*args):
    group_name, comparison1_player = decode_selection(comparison1_dropdown.value)
    reference_player = reference_dropdown.value

    comparison2_dropdown.options = comparison2_options_for_group(reference_player, comparison1_player, group_name)
    comparison2_dropdown.value = None
    comparison2_dropdown.disabled = group_name is None
    clear_outputs_after_selection_change()
    update_generate_button_state()

    with status_output:
        clear_output(wait=True)
        if group_name is None:
            print(f"Referenzspieler: {reference_player}")
            print("Bitte Vergleich 1 auswählen.")
        else:
            group_df = get_group_df_by_name(group_name)
            positions = sorted(group_df[POSITION_COL].dropna().unique())
            print(f"Referenzspieler: {reference_player}")
            print(f"Fixierte Gruppe: {group_name} | Positionen: {', '.join(positions)}")
            print(f"Vergleich 1: {comparison1_player}")
            print("Optional Vergleich 2 auswählen und dann 'generieren' klicken.")


def on_generate_clicked(button):
    with plot_output:
        clear_output(wait=True)
    with links_output:
        clear_output(wait=True)

    try:
        reference_player = reference_dropdown.value
        group_name, comparison1_player = decode_selection(comparison1_dropdown.value)
        group_name_2, comparison2_player = decode_selection(comparison2_dropdown.value)

        comparison_players = [comparison1_player]
        if comparison2_player is not None:
            if group_name_2 != group_name:
                raise ValueError("Vergleich 2 muss aus derselben Gruppe wie Vergleich 1 stammen.")
            comparison_players.append(comparison2_player)

        result = make_spider_plot_comparison(
            reference_player=reference_player,
            comparison_players=comparison_players,
            explicit_group=group_name,
            save_plot=save_checkbox.value,
        )

        with plot_output:
            display(result["figure"])
            plt.close(result["figure"])

        clickable_links_html = build_clickable_links_html(result["non_fcn_players"])
        if clickable_links_html:
            with links_output:
                display(IPyHTML(clickable_links_html))

        with status_output:
            clear_output(wait=True)
            print(f"Verwendete Gruppe: {result['group']}")
            print("Der Export enthält den kompletten Infobereich als Grafik.")
            print("Die wirklich klickbaren Transfermarkt-Links stehen zusätzlich direkt unter dem Plot.")

    except Exception as exc:
        with status_output:
            clear_output(wait=True)
            print(f"Fehler: {exc}")


reference_dropdown.observe(refresh_comparison1_options, names="value")
comparison1_dropdown.observe(refresh_comparison2_options, names="value")
generate_button.on_click(on_generate_clicked)

controls = widgets.HBox([
    reference_dropdown,
    comparison1_dropdown,
    comparison2_dropdown,
    widgets.VBox([generate_button, save_checkbox]),
])

# Initial befüllen.
refresh_comparison1_options()

display(controls, status_output, plot_output, links_output)


Output()

Output()

Output()